# EMOS Coding Lab: Automatic Coding with Text Similarity

In this lab we build a small automatic coding system step by step.

The task is simple to state: given a short text query, suggest the most appropriate classification code.

We will focus on transparency rather than performance. At each step we ask: what information do we have, what transformation are we applying, and how can we validate the result?

## 1. Introduction

**Automatic coding** means assigning a classification code to a text description. In Official Statistics, this can support tasks such as coding economic activities, occupations, products, or causes of death.

A useful automatic coding system should not only output a code. It should also help us understand why that code was suggested, how confident the system is, and where human validation is needed.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
import matplotlib.pyplot as plt

from src.data_loading import load_labelled_queries, load_teaching_sample, load_ateco_descriptors, create_teaching_sample
from src.descriptors import build_descriptors
from src.embeddings import compute_tfidf_embeddings, compute_centroid_embeddings, load_embedding_model, compute_sentence_embeddings
from src.similarity import compute_similarity_matrix, get_top_k_predictions, add_predictions_to_queries
from src.evaluation import summarize_accuracy, get_error_examples

## 2. Data loading

We use labelled queries as examples of texts that need to be coded. Each query already has a true code, so we can evaluate our automatic suggestions.

For the classroom exercise we use a small reproducible sample prepared from the raw data. The sample keeps only multi-word queries and removes duplicated query/code pairs. This keeps the notebook fast and makes the examples easier to inspect.

In [ ]:
raw_queries = load_labelled_queries()
raw_queries.head()

In [ ]:
# The sample was created with create_teaching_sample(raw_queries, min_words=3, sample_size=500).
sample_queries = load_teaching_sample()
sample_queries.head(10)

In [ ]:
sample_queries.shape, sample_queries["true_code"].nunique()

## 3. Classification and descriptors

A classification system is a structured list of possible codes. A code is not just a number: it represents a statistical category.

To use text similarity, we need text descriptions for the target codes. Here there is one important modelling choice: the labelled data is at 5-digit level, while the ATECO descriptor file contains more detailed 6-digit descriptors.

We test two simple strategies:

- `CONCAT`: combine all detailed descriptors belonging to the same 5-digit code into one text.
- `CENTROID`: embed each detailed descriptor separately, then average the vectors for each 5-digit code.

In [ ]:
DESCRIPTOR_METHOD = "CONCAT"  # Try also: "CENTROID"

raw_descriptors = load_ateco_descriptors()
descriptor_table = build_descriptors(raw_descriptors, method=DESCRIPTOR_METHOD)

# We keep the full set of candidate target codes. This is important:
# in a real coding task, the system should not know the true label in advance.
sample_queries = sample_queries[
    sample_queries["true_code"].isin(descriptor_table["code"])
].reset_index(drop=True)

descriptor_table.head()

In [ ]:
descriptor_table.shape

## 4. Embeddings

An **embedding** is a numerical representation of text. Intuitively, it places texts in a space where texts with similar meaning should be closer together.

Many modern embedding models are shared through **Hugging Face**, an online hub for machine learning models, datasets, and demos. In practice, this means we do not need to train a language model from scratch: we can reuse a pre-trained multilingual model and apply it to our coding task.

For reliability in a classroom setting, we start with a simple TF-IDF representation. It is not a modern semantic embedding model, but it makes the same similarity workflow visible and runs quickly.

If the sentence-transformers model is installed and available, the optional cell below switches to a multilingual semantic embedding model from Hugging Face.

In [ ]:
query_vectors, descriptor_vectors = compute_tfidf_embeddings(
    sample_queries["query"],
    descriptor_table["descriptor_text"],
)

if DESCRIPTOR_METHOD == "CENTROID":
    target_codes, target_vectors = compute_centroid_embeddings(
        descriptor_vectors,
        descriptor_table["code"],
    )
else:
    target_codes = descriptor_table["code"].tolist()
    target_vectors = descriptor_vectors

query_vectors.shape, target_vectors.shape, len(target_codes)

In [ ]:
# Optional: use a multilingual sentence embedding model instead of TF-IDF.
# The selected model is downloaded from Hugging Face the first time it is used.
# Larger models may need more memory, more time, or a GPU/cloud environment.

EMBEDDING_MODELS = [
    {
        "model_id": "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2",
        "trust_remote_code": False,
        "note": "small classroom default",
    },
    {
        "model_id": "BAAI/bge-m3",
        "trust_remote_code": False,
        "note": "strong multilingual embedding model",
    },
    {
        "model_id": "intfloat/multilingual-e5-base",
        "trust_remote_code": False,
        "note": "multilingual E5 base model",
    },
    {
        "model_id": "Alibaba-NLP/gte-multilingual-base",
        "trust_remote_code": True,
        "note": "requires trusting custom model code",
    },
    {
        "model_id": "Qwen/Qwen3-Embedding-4B",
        "trust_remote_code": True,
        "note": "larger model; better suited to GPU/cloud testing",
    },
]

selected_model = EMBEDDING_MODELS[0]
selected_model

In [ ]:
# Uncomment this cell to replace TF-IDF vectors with sentence embeddings.
#
# model = load_embedding_model(
#     selected_model["model_id"],
#     trust_remote_code=selected_model["trust_remote_code"],
# )
# query_vectors = compute_sentence_embeddings(sample_queries["query"], model)
# descriptor_vectors = compute_sentence_embeddings(descriptor_table["descriptor_text"], model)
#
# if DESCRIPTOR_METHOD == "CENTROID":
#     target_codes, target_vectors = compute_centroid_embeddings(
#         descriptor_vectors,
#         descriptor_table["code"],
#     )
# else:
#     target_codes = descriptor_table["code"].tolist()
#     target_vectors = descriptor_vectors

## 5. Similarity-based coding

Cosine similarity compares the direction of two vectors. In this lab, a high value means that a query and a target code descriptor use similar language or have similar meaning.

For each query, we rank all candidate codes by similarity and keep the best suggestions.

In [ ]:
similarity_matrix = compute_similarity_matrix(query_vectors, target_vectors)

top_k = get_top_k_predictions(
    similarity_matrix,
    target_codes,
    k=5,
)

predictions = add_predictions_to_queries(sample_queries, top_k)
predictions[["query", "true_code", "predicted_code", "confidence", "top_codes"]].head(10)

## 6. Evaluation

**Top-1 accuracy** asks: did the first suggested code match the labelled code?

**Top-k accuracy** asks: was the labelled code somewhere in the first k suggestions?

Top-k accuracy is important in Official Statistics because an automatic system may support a human coder by producing a shortlist rather than making a final decision alone.

In [ ]:
accuracy_table = summarize_accuracy(predictions, ks=(1, 3, 5))
accuracy_table

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))

ax.bar(
    accuracy_table["metric"],
    accuracy_table["value"],
    color="#3A7CA5",
)

ax.set_ylim(0, 1)
ax.set_ylabel("Accuracy")
ax.set_title("Top-k accuracy")

# Showing percentages makes the metric easier to read at a glance.
for index, value in enumerate(accuracy_table["value"]):
    ax.text(index, value + 0.02, f"{value:.1%}", ha="center")

plt.show()

## 7. Error analysis

Accuracy gives a summary, but error analysis teaches us more. We inspect cases where the labelled code was not the first suggestion.

For each error, ask: is the query ambiguous, is the descriptor too general, or are the two target codes genuinely similar?

In [ ]:
get_error_examples(predictions, k=1, n=15)

In [ ]:
# Cases where the first suggestion is wrong, but the correct code appears in the Top-5.
almost_right = predictions[
    (predictions["true_code"] != predictions["predicted_code"])
    & (predictions.apply(lambda row: row["true_code"] in row["top_codes"][:5], axis=1))
]

almost_right[["query", "true_code", "predicted_code", "top_codes"]].head(15)

## 8. Discussion: Official Statistics perspective

This lab shows the core mechanism behind similarity-based automatic coding. The workflow is transparent:

1. Define the target classification.
2. Describe each target code with text.
3. Represent queries and target codes as vectors.
4. Compare them with similarity.
5. Suggest the closest codes.
6. Evaluate and inspect errors.

In production, we would need more checks: data quality controls, monitoring over time, careful validation by domain experts, and clear rules for low-confidence cases.

The important idea is not that AI replaces statistical expertise. It can help organize work, produce candidate codes, and highlight uncertain cases where human judgment is most valuable.